# Provision Block Storage — proj03

Creates a **20 GiB Cinder block volume** on **KVM@TACC** for persisting small application state (PostgreSQL, RabbitMQ, etc.) across VM teardowns.

**Run this ONCE.** The volume survives VM teardowns — do not delete it unless decommissioning the project.

**Resources created:**
- Cinder volume: `block-<username>-proj03` (20 GiB, KVM@TACC)

**Next step:** Run `provision_vm.ipynb` to create the VM and attach this volume.

In [ ]:
from chi import context
import chi, os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
project = "proj03"

print(f"User: {username} | Project: {project} | Site: KVM@TACC")

In [ ]:
cinder_client = chi.clients.cinder()
volume_name = f"block-{username}-{project}"

# Idempotent — skip creation if volume already exists
existing = [v for v in cinder_client.volumes.list() if v.name == volume_name]
if existing:
    volume = existing[0]
    print(f"Volume '{volume_name}' already exists (status: {volume.status}) — skipping creation.")
else:
    volume = cinder_client.volumes.create(name=volume_name, size=20)
    print(f"Volume '{volume_name}' created (20 GiB). Waiting for AVAILABLE...")
    while True:
        volume = cinder_client.volumes.get(volume.id)
        if volume.status == "available":
            break
        time.sleep(3)
    print(f"Volume '{volume_name}' is AVAILABLE.")

print(f"\nVolume ID: {volume.id}")
print("Run provision_vm.ipynb next — it will attach this volume to the server.")